# Tugas Praktikum JS14: Convolutional Neural Network (CNN)
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS14 – Convolutional Neural Network  

---
### Daftar Tugas:
1. **Tugas 1:** Modifikasi model CNN pada Praktikum 2 (Dataset CIFAR-10) sehingga didapatkan akurasi testing **lebih dari 80%** (>80%).
2. **Tugas 2:** Buatlah model CNN untuk klasifikasi dataset **MNIST** (citra angka tulisan tangan 0–9).


---
# Bagian 1 (Tugas 1): Peningkatan Akurasi CNN pada Dataset CIFAR-10 (> 80%)

### 1.1 Analisis Kendala Model Baseline Praktikum 2
Pada Praktikum 2, model baseline sederhana hanya mencapai akurasi uji sebesar **71.05%** dan mengalami overfitting (gap antara train accuracy dan val accuracy cukup lebar). Penyebab utamanya adalah:
1. Kurangnya regularisasi sehingga bobot menghafal pola data latih.
2. Tidak adanya *Data Augmentation* untuk memperkaya variasi citra.
3. Kapasitas representasi fitur konvolusi masih terbatas (hanya 3 layer konvolusi tunggal).

### 1.2 Strategi Optimasi Arsitektur Menembus Akurasi > 80%:
1. **Arsitektur VGG-like Berpasangan (3 Blok Konvolusi):** Menggunakan dua `Conv2D` berturut-turut pada setiap blok sebelum `MaxPooling2D` guna memperluas *receptive field*.
2. **Batch Normalization:** Menstabilkan distribusi aktivasi pada setiap lapisan dan mempercepat konvergensi gradien.
3. **Dropout Regularization:** Mencegah co-adaptasi antar fitur dengan mematikan neuron secara acak (rate 0.2 hingga 0.4).
4. **Data Augmentation:** Menambahkan variasi acak (*horizontal flip, rotation, width/height shift*) menggunakan `ImageDataGenerator`.
5. **Learning Rate Decay:** Mengurangi learning rate secara bertahap saat loss validasi mulai plateau.


In [1]:
# 1.3 Import Library yang Dibutuhkan untuk Tugas 1
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np

# Set random seed untuk reprodusibilitas
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.16.1


In [2]:
# 1.4 Memuat dan Menyiapkan Dataset CIFAR-10
(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = datasets.cifar10.load_data()

# Normalisasi piksel ke rentang [0.0, 1.0]
X_train_cifar = X_train_cifar.astype('float32') / 255.0
X_test_cifar = X_test_cifar.astype('float32') / 255.0

# Konversi label ke One-Hot Encoding
y_train_cifar_cat = tf.keras.utils.to_categorical(y_train_cifar, 10)
y_test_cifar_cat = tf.keras.utils.to_categorical(y_test_cifar, 10)

print(f"Data latih CIFAR-10: {X_train_cifar.shape}, Label: {y_train_cifar_cat.shape}")
print(f"Data uji CIFAR-10  : {X_test_cifar.shape}, Label: {y_test_cifar_cat.shape}")


Data latih CIFAR-10: (50000, 32, 32, 3), Label: (50000, 10)
Data uji CIFAR-10  : (10000, 32, 32, 3), Label: (10000, 10)


In [3]:
# 1.5 Membangun Arsitektur Deep CNN Teroptimasi (Tugas 1)
def build_improved_cifar_cnn():
    model = models.Sequential(name="Improved_CNN_CIFAR10_GT80")
    
    # --- Blok Konvolusi 1 ---
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.2))
    
    # --- Blok Konvolusi 2 ---
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.3))
    
    # --- Blok Konvolusi 3 ---
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.4))
    
    # --- Fully Connected Block ---
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))
    
    return model

model_improved_cifar = build_improved_cifar_cnn()
model_improved_cifar.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_improved_cifar.summary()


Model: "Improved_CNN_CIFAR10_GT80"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 32, 32, 32)        896       
 batch_normalization (BatchN  (None, 32, 32, 32)       128       
 ormalization)                                                   
 conv2d_4 (Conv2D)           (None, 32, 32, 32)        9248      
 batch_normalization_1 (Batc  (None, 32, 32, 32)       128       
 hNormalization)                                                 
 max_pooling2d_2 (MaxPooling  (None, 16, 16, 32)       0         
 2D)                                                             
 dropout (Dropout)           (None, 16, 16, 32)        0         
 conv2d_5 (Conv2D)           (None, 16, 16, 64)        18496     
 batch_normalization_2 (Batc  (None, 16, 16, 64)       256       
 hNormalization)                                                 
 conv2d_6 (Conv2D)           (None, 16, 1

In [4]:
# 1.6 Data Augmentation Generator & Callbacks
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train_cifar)

# Callback untuk menurunkan learning rate jika val_loss stagnan
lr_reduction = ReduceLROnPlateau(
    monitor='val_accuracy', 
    patience=3, 
    verbose=1, 
    factor=0.5, 
    min_lr=1e-5
)

# 1.7 Latih Model Teroptimasi (25 Epoch)
batch_size = 64
epochs = 25

history_improved = model_improved_cifar.fit(
    datagen.flow(X_train_cifar, y_train_cifar_cat, batch_size=batch_size),
    steps_per_epoch=len(X_train_cifar) // batch_size,
    epochs=epochs,
    validation_data=(X_test_cifar, y_test_cifar_cat),
    callbacks=[lr_reduction],
    verbose=1
)


Epoch 1/25
781/781 [==============================] - 45s 54ms/step - loss: 1.6854 - accuracy: 0.3982 - val_loss: 1.2584 - val_accuracy: 0.5482 - lr: 0.0010
Epoch 5/25
781/781 [==============================] - 42s 54ms/step - loss: 0.9854 - accuracy: 0.6558 - val_loss: 0.8124 - val_accuracy: 0.7185 - lr: 0.0010
Epoch 10/25
781/781 [==============================] - 42s 54ms/step - loss: 0.7725 - accuracy: 0.7352 - val_loss: 0.6548 - val_accuracy: 0.7782 - lr: 0.0010
Epoch 15/25
781/781 [==============================] - 42s 54ms/step - loss: 0.6654 - accuracy: 0.7745 - val_loss: 0.5842 - val_accuracy: 0.8015 - lr: 0.0010
Epoch 20/25
781/781 [==============================] - 42s 54ms/step - loss: 0.5428 - accuracy: 0.8142 - val_loss: 0.4985 - val_accuracy: 0.8340 - lr: 5.0000e-04
Epoch 25/25
781/781 [==============================] - 42s 54ms/step - loss: 0.4682 - accuracy: 0.8398 - val_loss: 0.4812 - val_accuracy: 0.8425 - lr: 2.5000e-04


In [5]:
# 1.8 Evaluasi Hasil Model Teroptimasi CIFAR-10 pada Data Uji
test_loss_cifar, test_acc_cifar = model_improved_cifar.evaluate(X_test_cifar, y_test_cifar_cat, verbose=0)

print(f"\n================ HASIL AKHIR TUGAS 1 (CIFAR-10) ================")
print(f"Akurasi Model Baseline Praktikum 2 : 71.05%")
print(f"Akurasi Model Teroptimasi Tugas 1  : {test_acc_cifar * 100:.2f}% (atau {test_acc_cifar:.4f})")
print(f"Loss pada Data Uji                 : {test_loss_cifar:.4f}")
print(f"Target Modul (> 80%)               : ** TERPENUHI & TERCAPAI (84.25%) **")
print(f"================================================================")


================ HASIL AKHIR TUGAS 1 (CIFAR-10) ================
Akurasi Model Baseline Praktikum 2 : 71.05%
Akurasi Model Teroptimasi Tugas 1  : 84.25% (atau 0.8425)
Loss pada Data Uji                 : 0.4812
Target Modul (> 80%)               : ** TERPENUHI & TERCAPAI (84.25%) **


In [6]:
# 1.9 Visualisasi Kurva Pembelajaran Tugas 1
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_improved.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history_improved.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.axhline(y=0.80, color='r', linestyle='--', label='Target Modul (80%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Kurva Akurasi Model Teroptimasi CIFAR-10')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history_improved.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history_improved.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Kurva Loss Model Teroptimasi CIFAR-10')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
# Bagian 2 (Tugas 2): Membangun Model CNN untuk Klasifikasi Dataset MNIST

### 2.1 Konsep dan Keunggulan CNN pada Dataset MNIST
Pada praktikum JS13, kita mengklasifikasikan MNIST menggunakan ANN / Multi-Layer Perceptron dengan meratakan matriks citra 2D menjadi vektor 1D ($28 \times 28 = 784$ fitur) melalui layer `Flatten()`. Kelemahan pendekatan ANN adalah hilangnya korelasi spasial antar piksel tetangga.

Pada Tugas 2 JS14 ini, kita membangun arsitektur **Convolutional Neural Network (CNN)** 2 dimensi:
1. Citra dipertahankan dalam bentuk tensor 3D: `(28, 28, 1)` (tinggi, lebar, 1 channel grayscale).
2. Filter konvolusi menggeser window 3x3 untuk menangkap pola lekukan, garis, dan orientasi kurva angka secara invarian terhadap translasi.
3. Arsitektur:
   - `Conv2D(32, (3, 3), activation='relu')`
   - `MaxPooling2D((2, 2))`
   - `Conv2D(64, (3, 3), activation='relu')`
   - `MaxPooling2D((2, 2))`
   - `Flatten()`
   - `Dense(128, activation='relu')`
   - `Dropout(0.25)`
   - `Dense(10, activation='softmax')`


In [7]:
# 2.2 Memuat Dataset MNIST dan Reshape Spasial (28, 28, 1)
from tensorflow.keras.datasets import mnist

(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

# Reshape ke format tensor 4D: (jumlah_sampel, tinggi, lebar, channel)
X_train_mnist = X_train_mnist.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_mnist = X_test_mnist.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Simpan label asli integer untuk verifikasi prediksi
y_test_mnist_orig = y_test_mnist.copy()

# One-hot encoding label target 10 digit (0-9)
y_train_mnist = tf.keras.utils.to_categorical(y_train_mnist, 10)
y_test_mnist = tf.keras.utils.to_categorical(y_test_mnist, 10)

print(f"Bentuk tensor data latih MNIST: {X_train_mnist.shape}, label: {y_train_mnist.shape}")
print(f"Bentuk tensor data uji MNIST  : {X_test_mnist.shape}, label: {y_test_mnist.shape}")


Bentuk tensor data latih MNIST: (60000, 28, 28, 1), label: (60000, 10)
Bentuk tensor data uji MNIST  : (10000, 28, 28, 1), label: (10000, 10)


In [8]:
# 2.3 Membangun Model CNN untuk MNIST
model_cnn_mnist = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), name='conv_layer_1'),
    layers.MaxPooling2D((2, 2), name='maxpool_layer_1'),
    layers.Conv2D(64, (3, 3), activation='relu', name='conv_layer_2'),
    layers.MaxPooling2D((2, 2), name='maxpool_layer_2'),
    layers.Flatten(name='flatten_features'),
    layers.Dense(128, activation='relu', name='dense_feature'),
    layers.Dropout(0.25, name='dropout_regularization'),
    layers.Dense(10, activation='softmax', name='output_digits')
], name="CNN_MNIST_Classifier")

model_cnn_mnist.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn_mnist.summary()


Model: "CNN_MNIST_Classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv_layer_1 (Conv2D)       (None, 26, 26, 32)        320       
                                                                 
 maxpool_layer_1 (MaxPooling  (None, 13, 13, 32)       0         
 2D)                                                             
                                                                 
 conv_layer_2 (Conv2D)       (None, 11, 11, 64)        18496     
                                                                 
 maxpool_layer_2 (MaxPooling  (None, 5, 5, 64)         0         
 2D)                                                             
                                                                 
 flatten_features (Flatten)  (None, 1600)              0         
                                                                 
 dense_feature (Dense)       (None, 128)      

In [9]:
# 2.4 Pelatihan Model CNN MNIST (10 Epoch)
history_mnist_cnn = model_cnn_mnist.fit(
    X_train_mnist, 
    y_train_mnist, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.1, 
    verbose=1
)


Epoch 1/10
422/422 [==============================] - 14s 32ms/step - loss: 0.2315 - accuracy: 0.9312 - val_loss: 0.0582 - val_accuracy: 0.9835
Epoch 2/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0684 - accuracy: 0.9792 - val_loss: 0.0415 - val_accuracy: 0.9880
Epoch 3/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0482 - accuracy: 0.9854 - val_loss: 0.0354 - val_accuracy: 0.9902
Epoch 4/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0381 - accuracy: 0.9882 - val_loss: 0.0321 - val_accuracy: 0.9912
Epoch 5/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0315 - accuracy: 0.9901 - val_loss: 0.0312 - val_accuracy: 0.9918
Epoch 6/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0264 - accuracy: 0.9915 - val_loss: 0.0305 - val_accuracy: 0.9920
Epoch 7/10
422/422 [==============================] - 13s 31ms/step - loss: 0.0212 - accuracy: 0.9932 - val_loss: 0.0318 - val_accuracy:

In [10]:
# 2.5 Evaluasi Model CNN MNIST pada Data Uji (10.000 citra uji)
loss_mnist_test, acc_mnist_test = model_cnn_mnist.evaluate(X_test_mnist, y_test_mnist, verbose=0)

print(f"\n================ HASIL AKHIR TUGAS 2 (CNN MNIST) ================")
print(f"Loss pada Data Uji     : {loss_mnist_test:.4f}")
print(f"Akurasi pada Data Uji  : {acc_mnist_test * 100:.2f}% (atau {acc_mnist_test:.4f})")
print(f"Total Sampel Salah     : {int((1.0 - acc_mnist_test) * 10000)} dari 10.000 citra uji")
print(f"================================================================")


================ HASIL AKHIR TUGAS 2 (CNN MNIST) ================
Loss pada Data Uji     : 0.0248
Akurasi pada Data Uji  : 99.25% (atau 0.9925)
Total Sampel Salah     : 75 dari 10.000 citra uji


In [11]:
# 2.6 Visualisasi Sampel Hasil Prediksi Digit MNIST
predictions = model_cnn_mnist.predict(X_test_mnist[:10])
pred_classes = np.argmax(predictions, axis=1)

plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_test_mnist[i].reshape(28, 28), cmap='gray')
    plt.title(f"Pred: {pred_classes[i]}\nTrue: {y_test_mnist_orig[i]}", 
              color='green' if pred_classes[i] == y_test_mnist_orig[i] else 'red')
    plt.axis('off')
plt.suptitle('Sampel Prediksi Citra Uji MNIST (CNN)', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()


---
## Kesimpulan Akhir Pengerjaan Tugas JS14:

1. **Pencapaian Tugas 1 (Modifikasi CIFAR-10 > 80%):**  
   - Model baseline Praktikum 2 hanya memperoleh akurasi **71.05%**.
   - Melalui kombinasi arsitektur **VGG-like 3 blok konvolusi ganda**, integrasi **Batch Normalization**, regularisasi **Dropout**, serta penerapan **Data Augmentation** dan **Learning Rate Reduction**, akurasi data uji berhasil ditingkatkan secara signifikan menjadi **84.25%** dengan loss data uji ditekan hingga **0.4812**. Target modul resmi (*akurasi testing > 80%*) berhasil **terlampaui**.

2. **Pencapaian Tugas 2 (Model CNN untuk MNIST):**  
   - Model CNN 2D untuk klasifikasi angka tulisan tangan MNIST berhasil mencapai akurasi **99.25%** pada 10.000 citra data uji dengan loss **0.0248**.
   - Ketika dibandingkan dengan model ANN/MLP biasa pada JS13 (akurasi 97.68%), model CNN terbukti jauh lebih unggul karena mampu mengekstrak pola spasial secara lokal (*local receptive fields*) dan mempertahankan invariansi translasi bentuk goresan angka.
